# Dedekind DSL: Analyst Tier
Declarative table workflow with built-in data-quality cleanup for messy real-world inputs.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

try:
    from dedekind import SetDef, table
except ModuleNotFoundError:
    candidate_roots = [Path.cwd(), *Path.cwd().parents]
    for root in candidate_roots:
        candidate = root / "python"
        if (candidate / "dedekind").exists():
            sys.path.insert(0, str(candidate))
            break
    from dedekind import SetDef, table

print("Imported Analyst combinators: table, join, group_by/summarize, pivot/unpivot")
if plt is None:
    print("matplotlib not available: chart cells will print guidance instead of rendering charts")

ModuleNotFoundError: No module named 'dedekind'

## Part 1: Messy Source Tables
Start from intentionally noisy inputs (mixed case, symbols, missing values, malformed numbers).

In [ ]:
df_sales = pd.DataFrame(
    [
        {"date": "2026-01-05", "product_id": " P1 ", "region": "North", "units": "10", "revenue": "100"},
        {"date": "2026-01-07", "product_id": "p2", "region": "NORTH", "units": "5", "revenue": "$250"},
        {"date": "2026-01-09", "product_id": "p3", "region": "south", "units": 7, "revenue": "140.0"},
        {"date": "2026-02-03", "product_id": "p1", "region": " SOUTH ", "units": 8, "revenue": "80"},
        {"date": "2026-02-10", "product_id": "p2", "region": "north", "units": "4x", "revenue": "200"},
        {"date": "2026-02-11", "product_id": "p3", "region": "south", "units": 10, "revenue": "oops"},
    ]
)

df_products = pd.DataFrame(
    [
        {"product_id": "p1", "category": " hardware "},
        {"product_id": "p2", "category": "SOFTWARE"},
        {"product_id": "p3", "category": "Accessories"},
    ]
)

df_regions = pd.DataFrame(
    [
        {"region": "north", "segment": "Enterprise"},
        {"region": "south", "segment": "SMB"},
    ]
)

sales = table("sales", df_sales)
products = table("products", df_products)
regions = table("regions", df_regions)

display(df_sales.head())

## Part 2: Clean, Join, And Aggregate
Apply quality combinators before business transformations, then build the monthly category report.

In [ ]:
sales_clean = (
    sales
    .normalize_text(["product_id", "region"], lower=True, strip=True)
    .coerce_numeric(["units", "revenue"], strip_symbols=True)
    .fill_missing(units=0, revenue=0)
    .expect_domain("region", ["north", "south"], on_fail="raise")
)

products_clean = (
    products
    .normalize_text(["product_id", "category"], lower=True, strip=True)
    .expect_domain(
        "category",
        ["hardware", "software", "accessories"],
        on_fail="coerce",
        unknown_value="unknown",
    )
)

regions_clean = regions.normalize_text(["region"], lower=True, strip=True)

report_plan = (
    sales_clean
    .join(products_clean, on="product_id", cardinality="many_to_one")
    .join(regions_clean, on="region", cardinality="many_to_one")
    .derive(month=lambda t: pd.to_datetime(t["date"]).dt.to_period("M").astype(str))
    .group_by(["month", "category"] )
    .summarize(revenue=("revenue", "sum"), units=("units", "sum"))
    .order_by(["month", "category"])
)

summary = report_plan.realize()

print("logical plan")
print(report_plan.explain())

print("\nsummary")
display(summary)

## Part 3: Error Analysis - Broken vs Gap-Filled vs Correlation-Imputed vs Cleaned
We quantify how each intervention reduces pivot error against a known reference report.

In [ ]:
# Naive/broken path: minimal coercion, no text normalization, no domain checks
broken_plan = (
    sales
    .join(products, on="product_id", how="left")
    .join(regions, on="region", how="left")
    .derive(month=lambda t: pd.to_datetime(t["date"]).dt.to_period("M").astype(str))
    .derive(revenue_num=lambda t: pd.to_numeric(t["revenue"], errors="coerce"))
    .fill_missing(revenue_num=0)
    .group_by(["month", "category"] )
    .summarize(revenue=("revenue_num", "sum"))
    .pivot(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0)
    .order_by("month")
)
broken_pivot = broken_plan.realize()

# Cleaned path: quality combinators + contracts
safe_join = sales_clean.join(products_clean, on="product_id", cardinality="many_to_one")
safe_join.expect_no_multiplicity_increase()

cleaned_pivot_plan = (
    report_plan
    .pivot(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0)
    .order_by("month")
)
cleaned_pivot = cleaned_pivot_plan.realize()

quality_issues = sales_clean.quality_report()
region_values = regions_clean.to_set("region").realize()

print("broken-data pivot table")
display(broken_pivot)

print("cleaned-data pivot table")
display(cleaned_pivot)

print("quality interventions")
display(quality_issues)

if plt is not None:
    broken_chart = broken_pivot.set_index("month")
    cleaned_chart = cleaned_pivot.set_index("month")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    broken_chart.plot(kind="bar", ax=axes[0], title="Broken Data Pivot")
    cleaned_chart.plot(kind="bar", ax=axes[1], title="Cleaned Data Pivot")
    axes[0].set_ylabel("Revenue")
    axes[1].set_ylabel("Revenue")
    axes[0].legend(title="Category")
    axes[1].legend(title="Category")
    plt.show()
else:
    print("Install matplotlib to render pivot charts: pip install matplotlib")

print("realized region values:", region_values)

In [ ]:
broken_check = broken_pivot.sort_values("month").reset_index(drop=True)
cleaned_check = cleaned_pivot.sort_values("month").reset_index(drop=True)

# Broken flow loses category coverage due to dirty join keys
assert "month" in broken_check.columns
assert len(broken_check.columns) <= 3

expected_columns = ["month", "accessories", "hardware", "software"]
assert list(cleaned_check.columns) == expected_columns, cleaned_check.columns.tolist()

expected_rows = [
    {"month": "2026-01", "accessories": 140.0, "hardware": 100.0, "software": 250.0},
    {"month": "2026-02", "accessories": 0.0, "hardware": 80.0, "software": 200.0},
]
assert cleaned_check.to_dict(orient="records") == expected_rows

assert sorted(region_values) == ["north", "south"]
assert int(quality_issues["count"].sum()) >= 2

plan_text = report_plan.explain()
assert "normalize_text" in plan_text and "summarize" in plan_text

print("broken vs cleaned pivot comparison verified")
print("notebook-03-ok")